# Overview: Quantum Resource Estimation with `bloch_paw`

This notebook introduces the **Bloch--UPAW** framework for quantum resource estimation of periodic materials. It explains the problem, the algorithm, and the code organization — no code execution required.

**Prerequisites:** Familiarity with density functional theory (DFT), second quantization, and the basics of fault-tolerant quantum computing (block encoding, quantum phase estimation).

## The Problem

Simulating the electronic structure of periodic materials on a quantum computer requires:

1. **A Hamiltonian representation** compatible with quantum algorithms — specifically, a *linear combination of unitaries* (LCU) decomposition.
2. **Concrete resource counts** — how many logical qubits and Toffoli gates are needed for a target accuracy?

For molecules in a Gaussian basis, these resource estimates are well-studied. But **periodic solids** pose additional challenges:
- The Hamiltonian lives on a k-point mesh in reciprocal space.
- The projector augmented wave (PAW) method introduces on-site correction tensors that must be handled in the LCU decomposition.
- The pair density has a plane-wave (soft) part and a PAW augmentation part, both contributing to the Hamiltonian one-norm λ.

The **Bloch--UPAW** framework addresses all of these by deriving a complete LCU decomposition for the PAW Hamiltonian on a Monkhorst-Pack k-mesh, together with analytical formulas for the Toffoli and qubit costs.

## The Hamiltonian One-Norm $\lambda$

The central quantity computed by this code is the **one-norm** $\lambda$ of the Hamiltonian in the LCU basis:

$$
\lambda = \underbrace{\sum_{k,i} |\varepsilon_i(k)|}_{\text{one-body}}
+ \frac{1}{4} \sum_{J} \sum_{Q} \left[
    \underbrace{\sum_{G} \xi^{(J)}_G(Q)}_{\text{soft (plane-wave)}}
    + \underbrace{\sum_a \sum_{r \le s} |\varepsilon^a_{rs}|\, \chi^{a,J}_{rs}(Q)}_{\text{PAW augmentation}}
\right]
$$

where:
- $\varepsilon_i(\mathbf{k})$ are eigenvalues of the effective one-body matrix $h'(\mathbf{k})$, defined as the raw one-body Hamiltonian minus a mean-field exchange correction (Section II of the paper)
- $\xi$ captures the soft (plane-wave) two-body contribution via SVD of the Fourier-transformed pair density (the SVD decomposition formula in Section III)
- $\chi$ captures the PAW on-site correction via eigendecomposition of the $C^a$ tensor (the PAW augmentation formula in Section III)

The one-norm $\lambda$ directly determines the **query complexity** of quantum phase estimation: the number of block-encoding queries scales as $\mathcal{O}(\lambda / \varepsilon_\text{QPE})$.

## The Pipeline

The code implements a four-stage pipeline:

```
                     Classical (runs once)                      Quantum cost analysis
              ┌──────────────────────────────┐           ┌──────────────────────────────┐
              │                              │           │                              │
  GPAW DFT   │   PawExtractor               │   HDF5    │   OneNormCalculator           │
  ─────────► │   • extract ρ̃, D^a, C^a, h  ├──────────►│   • FFT + SVD → ξ           │
  (converged │   • export to HDF5           │   file    │   • C^a eigendecomp → χ      │
   calc)      │                              │           │   • assemble λ               │
              └──────────────────────────────┘           └──────────┬───────────────────┘
                                                                    │
                                                                    │ λ, R_avg, R0
                                                                    ▼
                                                         ┌──────────────────────────────┐
                                           PawReader     │   ResourceEstimator           │
                                           • load from   │   • Toffoli count formula     │
                                             HDF5        │   • qubit count formula       │
                                           • lazy or     │   • QROAM optimization       │
                                             eager mode  └──────────────────────────────┘
```

Each stage maps to a Python class:

| Stage | Class | Input | Output |
|-------|-------|-------|--------|
| 1. Extract | `PawExtractor` | GPAW calculator | HDF5 file |
| 2. Read | `PawReader` | HDF5 file | numpy arrays |
| 3. One-norm | `OneNormCalculator` | arrays from reader | $\lambda$, rank statistics |
| 4. Resources | `ResourceEstimator` | HDF5 metadata + $\lambda$ | Toffoli/qubit counts |

## The Seven Steps of the One-Norm Calculation

The `OneNormCalculator` computes $\lambda$ through a seven-step pipeline. Each step has a dedicated method:

| Step | Method | Computes | Description |
|------|--------|----------|-------------|
| 1 | `compute_soft_modes()` | FFT $\tilde{\rho}$ → batched SVD → soft eigenvalues $f^{(J)}$ | Soft (plane-wave) two-body decomposition |
| 2 | `diagonalize_Ca()` | Eigendecompose $C^a$ → weights $\varepsilon^a_{rs}$, eigenvectors $O$ | PAW on-site correction eigendecomposition |
| 3 | `compute_paw_modes()` | Contract $D^a$ with $O$, batched SVD → PAW eigenvalues $f^{a,J}$ | PAW augmentation mode spectrum |
| 4 | `compute_xi()` | Accumulate $\xi^{(J)}(Q,G)$ from soft eigenvalues | Soft contribution to $\lambda$ |
| 5 | `compute_chi()` | Accumulate $\chi^{a,J}_{rs}(Q)$ from PAW eigenvalues | PAW contribution to $\lambda$ |
| 6 | `diagonalize_one_plus_two_body()` | Diagonalise effective one-body matrix → $\varepsilon_i(\mathbf{k})$ | One-body eigenvalues |
| 7 | `lambda_one_norm()` | Assemble $\lambda$ = term1 + term2 + term3 | Total one-norm assembly |

Steps 1--3 are the expensive SVD-based decompositions. Steps 4--5 are cheap accumulations. Step 6 is independent of steps 1--5 and handles the one-body contribution. Step 7 brings everything together.

Each step caches its results, so calling `lambda_one_norm()` automatically runs all preceding steps only once.

## Key Physical Quantities in the HDF5 File

The HDF5 file produced by `PawExtractor.export_hdf5()` contains:

| Dataset | Shape | Description |
|---------|-------|-------------|
| `rho_tilde/data` | (Nk, Nb, Nk, Nb, Nx, Ny, Nz) | Smooth pseudo pair-density $\tilde{\rho}$ on the FFT grid |
| `C_tensor/atom_NNNN` | (na, na, na, na) | PAW on-site Coulomb correction tensor $C^a$ |
| `D_tensor/atom_NNNN` | (Nk, Nb, Nk, Nb, na, na) | Projector density matrix $D^a$ |
| `one_body/H_kikj` | (Nk, Nb, Nk, Nb) | One-body matrix elements $h_{ki,k'j}$ |
| `kmesh/bz/cart_1_perA` | (Nk, 3) | Cartesian k-point coordinates (Å⁻¹) |
| `lattice/A_direct_ang` | (3, 3) | Primitive lattice vectors (Å) |
| `supercell_size` | (3,) | Supercell scaling factors (Lx, Ly, Lz) |
| `Npw` | scalar | Number of plane waves |

The pair density $\tilde{\rho}$ is by far the largest dataset — for a 2×2×2 k-mesh with 5 bands on a 24³ grid, it takes ~30 MB.

## Code Organization

```
bloch_paw/
  __init__.py          # Public API: PawReader, OneNormCalculator, ResourceEstimator
  extractor.py         # PawExtractor — requires GPAW (classical DFT)
  reader.py            # PawReader — reads HDF5, no GPAW dependency
  one_norm.py          # OneNormCalculator — 7-step λ computation
  resources.py         # ResourceEstimator — Toffoli/qubit formulas
```

**Design principle:** Only `extractor.py` depends on GPAW. The rest of the pipeline uses only `numpy` and `h5py`, so you can analyse pre-computed data on any machine without installing GPAW.

The package exposes three classes at the top level:
```python
from bloch_paw import PawReader, OneNormCalculator, ResourceEstimator
```

`PawExtractor` is imported separately when needed:
```python
from bloch_paw.extractor import PawExtractor
```

## Quick Start: Minimal Working Example

If you have an HDF5 file (e.g., from a collaborator or from `PawExtractor.export_hdf5()`), the analysis pipeline is just a few lines:

```python
from bloch_paw import PawReader, OneNormCalculator, ResourceEstimator

# Load data
reader = PawReader("integrals.h5")
inputs = reader.to_calculator_inputs()

# Compute one-norm
calc = OneNormCalculator(**inputs)
lam = calc.lambda_one_norm()
R_avg, R0 = calc.compute_average_rank()

# Estimate quantum resources
eps_chem = 1.6e-3       # chemical accuracy in Hartree (1 kcal/mol)
eps_qpe = eps_chem / 5  # QPE precision: 1/5 of error budget

est = ResourceEstimator.from_hdf5("integrals.h5")
toffolis = est.toffoli_count_per_be(Rl=R_avg, R0=R0)
qubits = est.total_qubits(Rl=R_avg, R0=R0, lam=lam, eps_qpe=eps_qpe)

print(f"One-norm λ = {lam:.2f}")
print(f"Toffoli gates per query: {toffolis:,}")
print(f"Logical qubits: {qubits:,}")
```

The following notebooks walk through each stage in detail with real data.

## Notebook Series

| Notebook | Purpose |
|----------|---------|
| **01_overview** (this notebook) | Conceptual introduction |
| **02_gpaw_extraction** | Run GPAW DFT, extract PAW data to HDF5 with `PawExtractor` |
| **03_reading_paw_data** | Load and explore HDF5 data with `PawReader` |
| **04_one_norm_calculation** | Step-by-step $\lambda$ computation with `OneNormCalculator` |
| **05_resource_estimation** | Toffoli and qubit counts with `ResourceEstimator` |
| **06_full_pipeline** | End-to-end workflow from GPAW to resource estimates |

All runnable notebooks use the test data at `data/lcbo_2x2x2.h5` --- a 2x2x2 k-mesh LCBO calculation for metallic hydrogen (H fcc, a=3.67 A, 5 bands, PBE functional).